In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [3]:

# 1. INSTALLATIONER (kör denna cell först)
!pip install -qqq "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git" --progress-bar off
from torch import __version__; from packaging.version import Version as V
xformers = "xformers==0.0.27" if V(__version__) < V("2.4.0") else "xformers"
!pip install -qqq --no-deps {xformers} trl peft accelerate bitsandbytes triton --progress-bar off

# 2. IMPORTS
import numpy as np
import torch
import unsloth
from unsloth.chat_templates import get_chat_template
from unsloth import FastLanguageModel, is_bfloat16_supported
from transformers import TrainingArguments,TextStreamer
import trl
from trl import SFTTrainer
import os
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
from datasets import load_dataset
import pandas as pd
print(trl.__version__) #0.24.0
import gc


  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
0.24.0


In [4]:

# ============================================================================
# HYPERBAND OPTIMIZER KLASS
# ============================================================================

class HyperbandOptimizer:
    """Hyperband för hyperparameteroptimering av LLM fine-tuning."""

    def __init__(self, max_resource=5, eta=3, seed=42):
        self.max_resource = max_resource
        self.eta = eta
        self.seed = seed
        np.random.seed(seed)

        # Beräkna antal brackets
        self.s_max = int(np.floor(np.log(max_resource) / np.log(eta)))
        self.B = (self.s_max + 1) * max_resource

        # Spara resultat
        self.results = []
        self.best_config = None
        self.best_loss = float('inf')

    def sample_configuration(self):
        """Sampla slumpmässig hyperparameterkonfiguration."""
        config = {
          'learning_rate': float(np.random.choice([1e-5, 3e-5, 5e-5, 1e-4])),
          'per_device_train_batch_size': int(np.random.choice([1, 2, 4])),
          'gradient_accumulation_steps': int(np.random.choice([1, 2, 4])),
          'weight_decay': float(np.random.uniform(0.0, 0.05)),
          'warmup_steps': int(np.random.choice([0, 10, 20])),
          'lr_scheduler_type': np.random.choice(['linear', 'cosine', 'constant']),
          'lora_r': int(np.random.choice([4, 8, 16])),
          'lora_alpha': int(np.random.choice([16, 32])),
          'lora_dropout': float(np.random.uniform(0.0, 0.1)),
        }
        return config

    def train_and_evaluate(self, config, resource, base_model_name, tokenizer,
                          train_dataset, eval_dataset, max_seq_length,
                          output_dir, is_bfloat16):
        """Träna och evaluera med given konfiguration."""

        # Ladda ny modell för varje config
        model, _ = FastLanguageModel.from_pretrained(
            model_name=base_model_name,
            max_seq_length=max_seq_length,
            load_in_4bit=True,
            dtype=None,
        )

        # Applicera PEFT med config
        model = FastLanguageModel.get_peft_model(
            model,
            r=config['lora_r'],
            lora_alpha=config['lora_alpha'],
            lora_dropout=config['lora_dropout'],
            target_modules=["q_proj", "k_proj", "v_proj", "up_proj",
                          "down_proj", "o_proj", "gate_proj"],
            use_rslora=True,
            use_gradient_checkpointing="unsloth"
        )

        # Training arguments
        training_args = TrainingArguments(
            learning_rate=config['learning_rate'],
            lr_scheduler_type=config['lr_scheduler_type'],
            per_device_train_batch_size=config['per_device_train_batch_size'],
            gradient_accumulation_steps=config['gradient_accumulation_steps'],
            max_steps=int(resource) * 8 ,
            fp16=not is_bfloat16,
            bf16=is_bfloat16,
            logging_steps=max(1, int(resource // 10)),
            optim="adamw_8bit",
            weight_decay=config['weight_decay'],
            warmup_steps=config['warmup_steps'],
            output_dir=output_dir,
            save_strategy="no",
            seed=0,
            report_to="none",
            remove_unused_columns=False,
            #eval_strategy="steps",
            #eval_steps=max(1, int(resource // 2)),
            load_best_model_at_end=False,
        )

        # Skapa trainer
        trainer = SFTTrainer(
            model=model,
            tokenizer=tokenizer,
            train_dataset=train_dataset,
            eval_dataset=eval_dataset,
            dataset_text_field="text",
            max_seq_length=max_seq_length,
            dataset_num_proc=2,
            packing=True,
            args=training_args,
        )

        # Träna och evaluera
        try:
            trainer.train()
            eval_results = trainer.evaluate()
            eval_loss = eval_results['eval_loss']
        except Exception as e:
            print(f"    Error during training: {e}")
            eval_loss = float('inf')

        # Rensa minne
        del eval_results
        del model, trainer
        gc.collect()
        torch.cuda.empty_cache()

        return eval_loss

    def successive_halving(self, n, r, s, base_model_name, tokenizer,
                          train_dataset, eval_dataset, max_seq_length,
                          output_dir, is_bfloat16):
        """Kör successive halving för en bracket."""

        # Generera initiala configs
        configs = [self.sample_configuration() for _ in range(n)]
        losses = [None] * n

        print(f"\n{'='*70}")
        print(f"  BRACKET s={s}: {n} configs, {int(r)} initial steps")
        print(f"{'='*70}")

        # Successive halving loop
        for i in range(s + 1):
            n_i = int(n * (self.eta ** (-i)))
            r_i = int(r * (self.eta ** i))

            print(f"\n  Round {i+1}/{s+1}: Training {len(configs)} configs for {int(r_i) * 8} steps")
            print(f"  {'-'*66}")

            for j, config in enumerate(configs):
                print(f"    Config {j+1}/{len(configs)}: LR={config['learning_rate']:.1e}, "
                      f"BS={config['per_device_train_batch_size']}, "
                      f"LoRA_r={config['lora_r']}", end=" ")

                loss = self.train_and_evaluate(
                    config, r_i, base_model_name, tokenizer, train_dataset,
                    eval_dataset, max_seq_length, output_dir, is_bfloat16
                )
                losses[j] = loss

                print(f"→ Loss: {loss:.4f}")

                # Spara resultat
                self.results.append({
                    'bracket': s,
                    'round': i,
                    'config': config.copy(),
                    'resource': r_i,
                    'loss': loss
                })

                # Uppdatera bästa
                if loss < self.best_loss:
                    self.best_loss = loss
                    self.best_config = config.copy()
                    print(f"      ★★★ NEW BEST! Loss: {loss:.4f} ★★★")

            # Behåll topp k configs
            if i < s:
                k = int(n_i / self.eta)
                indices = np.argsort(losses)[:k]
                configs = [configs[idx] for idx in indices]
                losses = [losses[idx] for idx in indices]
                print(f"\n  → Keeping top {k} configs")

        return self.best_config, self.best_loss

    def run(self, base_model_name, tokenizer, train_dataset, eval_dataset,
            max_seq_length, output_dir, is_bfloat16):
        """Kör hela Hyperband-algoritmen."""
        print(f"\n{'#'*70}")
        print(f"  HYPERBAND OPTIMIZATION START")
        print(f"  Max resource: {int(self.max_resource) * 8} steps | eta: {self.eta}")
        print(f"  Total brackets: {self.s_max + 1}")
        print(f"{'#'*70}")

        # Kör varje bracket
        for s in range(self.s_max, -1, -1):
            n = int(np.ceil((self.B / self.max_resource) *
                           (self.eta ** s) / (s + 1)))
            r = self.max_resource * (self.eta ** (-s))

            self.successive_halving(
                n, r, s, base_model_name, tokenizer, train_dataset,
                eval_dataset, max_seq_length, output_dir, is_bfloat16
            )

        print(f"\n{'#'*70}")
        print(f"  HYPERBAND OPTIMIZATION COMPLETE!")
        print(f"  Best loss: {self.best_loss:.4f}")
        print(f"  Best config:")
        for key, value in self.best_config.items():
            print(f"    {key}: {value}")
        print(f"{'#'*70}\n")

        return self.best_config, self.best_loss, self.results

# ============================================================================
# HUVUDFUNKTION
# ============================================================================

def run_hyperband():
    """Huvudfunktion som kör hela Hyperband-optimeringen."""

    print("Starting Hyperband optimization for LLM fine-tuning...")

    # -------------------------------------------------------------------------
    # STEG 1: Ladda base model och tokenizer
    # -------------------------------------------------------------------------
    print("\n[1/5] Loading base model...")
    max_seq_length = 1024
    base_model_name = "unsloth/Llama-3.2-1B-bnb-4bit"

    _, tokenizer = FastLanguageModel.from_pretrained(
        model_name=base_model_name,
        max_seq_length=max_seq_length,
        load_in_4bit=True,
        dtype=None,
    )

    tokenizer = get_chat_template(
        tokenizer,
        chat_template="chatml",
        mapping={"role": "from", "content": "value",
                "user": "human", "assistant": "gpt"}
    )

    # -------------------------------------------------------------------------
    # STEG 2: Ladda och förbered dataset
    # -------------------------------------------------------------------------
    print("[2/5] Loading dataset...")

    def apply_template(examples):
        messages = examples["conversations"]
        text = [tokenizer.apply_chat_template(message, tokenize=False,
                add_generation_prompt=False) for message in messages]
        return {"text": text}

    dataset = load_dataset("mlabonne/FineTome-100k", split="train[:10000]")
    dataset = dataset.shuffle(seed=42)

    # Använd mindre dataset för snabbare hyperband (ändra 5000 om du vill)
    small_dataset = dataset.select(range(800))
    train_and_test = small_dataset.train_test_split(test_size=0.1, seed=42)

    train_dataset = train_and_test["train"].map(apply_template, batched=True)
    eval_dataset = train_and_test["test"].map(apply_template, batched=True)

    print(f"  Train size: {len(train_dataset)}, Eval size: {len(eval_dataset)}")

    # -------------------------------------------------------------------------
    # STEG 3: Initiera Hyperband
    # -------------------------------------------------------------------------
    print("[3/5] Initializing Hyperband...")

    hyperband = HyperbandOptimizer(
        max_resource=9,  # Öka till 500-1000 för bättre resultat
        eta=3,
        seed=42
    )

    # -------------------------------------------------------------------------
    # STEG 4: Kör Hyperband-optimering
    # -------------------------------------------------------------------------
    print("[4/5] Running Hyperband optimization...")

    output_dir = "/content/drive/MyDrive/Scalable ML/Lab2/hyperbands"
    os.makedirs(output_dir, exist_ok=True)

    best_config, best_loss, results = hyperband.run(
        base_model_name=base_model_name,
        tokenizer=tokenizer,
        train_dataset=train_dataset,
        eval_dataset=eval_dataset,
        max_seq_length=max_seq_length,
        output_dir=output_dir,
        is_bfloat16=is_bfloat16_supported()
    )


    return best_config, best_loss, results

# ============================================================================
# KÖR OPTIMERINGEN
# ============================================================================

best_config, best_loss, results = run_hyperband()

Starting Hyperband optimization for LLM fine-tuning...

[1/5] Loading base model...
==((====))==  Unsloth 2025.11.6: Fast Llama patching. Transformers: 4.57.2.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.741 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.9.0+cu126. CUDA: 7.5. CUDA Toolkit: 12.6. Triton: 3.5.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.33.post2. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


model.safetensors:   0%|          | 0.00/1.03G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/230 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/459 [00:00<?, ?B/s]

...

→ Loss: 1.2701

  → Keeping top 1 configs

  Round 3/3: Training 1 configs for 72 steps
  ------------------------------------------------------------------
    Config 1/1: LR=1.0e-04, BS=2, LoRA_r=16 ==((====))==  Unsloth 2025.11.6: Fast Llama patching. Transformers: 4.57.2.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.741 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.9.0+cu126. CUDA: 7.5. CUDA Toolkit: 12.6. Triton: 3.5.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.33.post2. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


The model is already on multiple devices. Skipping the move to device specified in `args`.
==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 720 | Num Epochs = 1 | Total steps = 72
O^O/ \_/ \    Batch size per device = 2 | Gradient accumulation steps = 2
\        /    Data Parallel GPUs = 1 | Total batch size (2 x 2 x 1) = 4
 "-____-"     Trainable parameters = 11,272,192 of 1,247,086,592 (0.90% trained)


Step,Training Loss
1,1.550100
2,1.280800
3,1.214900
4,1.194000
5,0.997800
6,1.040200
7,1.276700
8,1.385200
9,1.181600
10,0.890000


    Error during training: CUDA out of memory. Tried to allocate 16.00 MiB. GPU 0 has a total capacity of 14.74 GiB of which 2.12 MiB is free. Process 3155 has 14.74 GiB memory in use. Of the allocated memory 14.49 GiB is allocated by PyTorch, and 98.03 MiB is reserved by PyTorch but unallocated. If reserved but unallocated memory is large try setting PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True to avoid fragmentation.  See documentation for Memory Management  (https://pytorch.org/docs/stable/notes/cuda.html#environment-variables)


UnboundLocalError: cannot access local variable 'eval_results' where it is not associated with a value

##Training (Kanske kan flyttas till Lab2.ipynb?Hänvisa till denna notebook för hyperparameter tuning. Och sen göra en hel träning i den andra notebooken).

In [ ]:
 # -------------------------------------------------------------------------
    # STEG 5: Träna final model med bästa config
    # -------------------------------------------------------------------------
    print("[5/5] Training final model with best configuration...")

    # Ladda modell med bästa config
    final_model, final_tokenizer = FastLanguageModel.from_pretrained(
        model_name=base_model_name,
        max_seq_length=max_seq_length,
        load_in_4bit=True,
        dtype=None,
    )

    final_model = FastLanguageModel.get_peft_model(
        final_model,
        r=best_config['lora_r'],
        lora_alpha=best_config['lora_alpha'],
        lora_dropout=best_config['lora_dropout'],
        target_modules=["q_proj", "k_proj", "v_proj", "up_proj",
                      "down_proj", "o_proj", "gate_proj"],
        use_rslora=True,
        use_gradient_checkpointing="unsloth"
    )

    # Använd hela datasetet för final training
    full_train = dataset.select(range(90000)).map(apply_template, batched=True)
    full_eval = dataset.select(range(90000, 100000)).map(apply_template, batched=True)

    final_output_dir = "/content/drive/MyDrive/Scalable ML/Lab2/checkpoints"
    os.makedirs(final_output_dir, exist_ok=True)

    final_trainer = SFTTrainer(
        model=final_model,
        tokenizer=final_tokenizer,
        train_dataset=full_train,
        eval_dataset=full_eval,
        dataset_text_field="text",
        max_seq_length=max_seq_length,
        dataset_num_proc=2,
        packing=True,
        args=TrainingArguments(
            learning_rate=best_config['learning_rate'],
            lr_scheduler_type=best_config['lr_scheduler_type'],
            per_device_train_batch_size=best_config['per_device_train_batch_size'],
            gradient_accumulation_steps=best_config['gradient_accumulation_steps'],
            max_steps=1610,  # Full training
            fp16=not is_bfloat16_supported(),
            bf16=is_bfloat16_supported(),
            logging_steps=10,
            optim="adamw_8bit",
            weight_decay=best_config['weight_decay'],
            warmup_steps=best_config['warmup_steps'],
            output_dir=final_output_dir,
            save_strategy="steps",
            save_steps=100,
            save_total_limit=3,
            seed=0,
            report_to="none",
            remove_unused_columns=False,
        ),
    )

    print("\nTraining final model...")
    final_trainer.train()

    # Spara resultat
    results_df = pd.DataFrame(results)
    results_df.to_csv(f"{final_output_dir}/hyperband_results.csv", index=False)

    print("\n" + "="*70)
    print("DONE! Final model saved to:", final_output_dir)
    print("="*70)